# NDT7 (M-Lab) Data Prep — Vietnam Broadband, Province x Quarter

Aggregates the junior-delivered `data/ndt7/vn/mlab_vn_clean.parquet` (24.7M raw NDT7 test
records, already ISP-classified and province-joined via per-IP lookup + point-in-polygon)
into the same province x quarter format used for Thailand, so it can plug directly into a
Ookla-vs-NDT7 comparison notebook.

Mirrors the Thailand NDT7 pipeline's methodology (`notebooks/ndt7/ndt7_edav2.ipynb`, cells
32-37): bin raw lat/lon points into Ookla-style zoom-16 slippy tiles first, then aggregate
tiles up to province level. This keeps `n_tiles`/`is_reliable` comparable across countries
instead of just averaging raw points directly to province (input data already has an exact
per-record province join from the junior's pipeline, which is *more* precise, but tiling
first preserves the same spatial-coverage reliability check Thailand and all four Ookla
country notebooks use: `total_tests >= 100 & n_tiles >= 5`).

Output: `data/exports/ndt7_vietnam_province_quarterly.csv` — same column names as
`data/exports/ookla_vietnam_province_quarterly.csv` for a trivial merge/concat in the
comparison notebook (no legacy Thailand-era rename dance).

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../data/ndt7/vn/mlab_vn_clean.parquet'
VN_REF_CSV  = '../../data/reference/vietnam_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

COLS = ['date', 'mean_throughput_mbps', 'min_rtt', 'latitude', 'longitude',
        'type', 'network_type', 'province']

### 1. Load & Filter — Broadband Only, Positive Throughput

In [2]:
pf = pq.ParquetFile(RAW_PARQUET)
print(f"Total rows in file: {pf.metadata.num_rows:,}")

chunks = []
for batch in pf.iter_batches(columns=COLS, batch_size=2_000_000):
    df = batch.to_pandas()
    df = df[(df['network_type'] == 'broadband') & (df['mean_throughput_mbps'] > 0)]
    df = df.dropna(subset=['latitude', 'longitude', 'province', 'date'])
    chunks.append(df)

raw = pd.concat(chunks, ignore_index=True)
del chunks
print(f"Broadband rows after filter: {len(raw):,} ({len(raw)/pf.metadata.num_rows:.1%} of total)")
raw.head()

Total rows in file: 24,667,548


Broadband rows after filter: 22,014,446 (89.2% of total)


,date,mean_throughput_mbps,min_rtt,latitude,longitude,type,network_type,province
0,2024-05-13,123.680848,59.628,9.0177,105.2856,upload,broadband,Cà Mau
1,2024-10-31,73.124190,46.448,9.1335,104.9415,upload,broadband,Cà Mau
2,2023-07-02,25.920424,58.196,9.1771,105.1464,download,broadband,Cà Mau
3,2023-01-04,20.879018,53.009,9.1771,105.1464,upload,broadband,Cà Mau
4,2023-12-22,4.699463,54.568,9.1771,105.1464,upload,broadband,Cà Mau


### 2. Quarter Labels (Ookla-Style `YYYY-QN`)

In [3]:
raw['date'] = pd.to_datetime(raw['date'])
raw['year_q'] = (
    raw['date'].dt.to_period('Q').astype(str)
    .str.replace(r'(\d{4})Q(\d)', r'\1-Q\2', regex=True)
)
raw['min_rtt'] = raw['min_rtt'].clip(upper=2000)
print(raw['year_q'].value_counts().sort_index())

year_q
2023-Q1      201306
2023-Q2      270516
2023-Q3      362507
2023-Q4      277587
2024-Q1      353221
2024-Q2     1888085
2024-Q3      694062
2024-Q4      445991
2025-Q1    11695952
2025-Q2     2986669
2025-Q3     1715060
2025-Q4     1123490
Name: count, dtype: int64


### 3. Zoom-16 Slippy Tile Assignment

Same tile scheme Ookla publishes its own tiles in — makes `n_tiles` comparable across Ookla and NDT7, and across countries.

In [4]:
lat_rad = np.radians(raw['latitude'].clip(-85.05112878, 85.05112878))
mercator_y = np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad))

raw['tile_x'] = ((raw['longitude'].astype(float) + 180) / 360 * N_TILES).astype(int).clip(0, N_TILES - 1)
raw['tile_y'] = ((1 - mercator_y / np.pi) / 2 * N_TILES).astype(int).clip(0, N_TILES - 1)
raw['tile_id'] = raw['tile_x'].astype(str) + '_' + raw['tile_y'].astype(str)
print(f"Assigned tiles for {len(raw):,} records, {raw['tile_id'].nunique():,} distinct tiles.")

Assigned tiles for 22,014,446 records, 856 distinct tiles.


### 4. Tile-Level Aggregation

Province comes straight from the input (junior's per-IP + point-in-polygon join is more precise than re-deriving it from a tile centroid) — take the majority province per tile as a sanity check, tiles are ~610m so this should almost always be unanimous.

In [5]:
tile_agg = raw.groupby(['year_q', 'tile_id', 'type']).agg(
    tile_mean=('mean_throughput_mbps', 'mean'),
    tile_median=('mean_throughput_mbps', 'median'),
    tile_lat=('min_rtt', 'mean'),
    test_count=('mean_throughput_mbps', 'count'),
    province=('province', lambda s: s.mode().iat[0]),
).reset_index()

tile_agg = tile_agg[tile_agg['test_count'] >= MIN_TILE_TESTS].copy()
print(f"Tile x quarter x type rows after MIN_TILE_TESTS>={MIN_TILE_TESTS} filter: {len(tile_agg):,}")

Tile x quarter x type rows after MIN_TILE_TESTS>=3 filter: 5,568


### 5. Province-Level Weighted Aggregation

In [6]:
dl = tile_agg[tile_agg['type'] == 'download']
ul = tile_agg[tile_agg['type'] == 'upload']

dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
    'avg_d_mbps': np.average(g['tile_mean'], weights=g['test_count']),
    'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['test_count']),
    'total_tests': g['test_count'].sum(),
    'n_tiles': g['tile_id'].nunique(),
}), include_groups=False).reset_index()

ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
    'avg_u_mbps': np.average(g['tile_mean'], weights=g['test_count']),
}), include_groups=False).reset_index()

master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
master = master.rename(columns={'year_q': 'quarter'})
master['year'] = master['quarter'].str.slice(0, 4).astype(int)
master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)
print(f"Province x quarter rows: {len(master)}")
master.head()

Province x quarter rows: 733


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1
0,2023-Q1,An Giang,29.349034,112.389486,759.0,10.0,31.547373,2023,1
1,2023-Q1,Bà Rịa–Vũng Tàu,35.333411,112.670980,1558.0,7.0,28.657948,2023,1
2,2023-Q1,Bình Dương,31.278105,107.479716,747.0,11.0,25.175450,2023,1
3,2023-Q1,Bình Phước,29.091671,124.483435,278.0,8.0,24.916346,2023,1
4,2023-Q1,Bình Thuận,31.337100,97.125557,314.0,5.0,29.936516,2023,1


### 6. Reliability Flag — Same Threshold as Ookla (`total_tests>=100 & n_tiles>=5`)

In [7]:
master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
reliable_by_q = master.groupby('quarter')['is_reliable'].agg(['sum', 'count'])
reliable_by_q['pct'] = (reliable_by_q['sum'] / reliable_by_q['count'] * 100).round(1)
print(reliable_by_q)
print(f"\nOverall reliable: {master['is_reliable'].sum()} / {len(master)} ({master['is_reliable'].mean():.1%})")

         sum  count   pct
quarter                  
2023-Q1   30     62  48.4
2023-Q2   29     62  46.8
2023-Q3   26     63  41.3
2023-Q4   20     61  32.8
2024-Q1   24     62  38.7
2024-Q2   16     61  26.2
2024-Q3   15     62  24.2
2024-Q4   15     61  24.6
2025-Q1   14     61  23.0
2025-Q2    8     62  12.9
2025-Q3    7     60  11.7
2025-Q4    6     56  10.7

Overall reliable: 210 / 733 (28.6%)


### 7. Merge Province Reference (Region, Tier, GDP, Density)

In [8]:
ref = pd.read_csv(VN_REF_CSV)
master = master.merge(
    ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
         'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
    left_on='province', right_on='province_en', how='left'
).drop(columns=['province_en'])

missing_ref = master[master['region'].isna()]['province'].unique()
print(f"Provinces with no reference match: {list(missing_ref)}")
master.head()

Provinces with no reference match: []


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,An Giang,29.349034,112.389486,759.0,10.0,31.547373,2023,1,True,Mekong Delta,1,2057000,3791.46,540,10591.12,338704.14
1,2023-Q1,Bà Rịa–Vũng Tàu,35.333411,112.670980,1558.0,7.0,28.657948,2023,1,True,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
2,2023-Q1,Bình Dương,31.278105,107.479716,747.0,11.0,25.175450,2023,1,True,Southeast,2,2564000,3663.54,901,10233.79,327276.61
3,2023-Q1,Bình Phước,29.091671,124.483435,278.0,8.0,24.916346,2023,1,True,Southeast,2,1313000,3606.56,145,10074.62,322186.39
4,2023-Q1,Bình Thuận,31.337100,97.125557,314.0,5.0,29.936516,2023,1,True,South Central Coast,2,1498000,3090.17,155,8632.13,276055.50


### 8. Export — Same Schema as `ookla_vietnam_province_quarterly.csv`

In [9]:
EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

out = master[EXPORT_COLS].copy()
OUT_PATH = '../../data/exports/ndt7_vietnam_province_quarterly.csv'
out.to_csv(OUT_PATH, index=False)
print(f"Exported {len(out)} rows -> {OUT_PATH}")
out.head(3)

Exported 733 rows -> ../../data/exports/ndt7_vietnam_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,An Giang,2023-Q1,2023,1,29.349034,31.547373,112.389486,759.0,10.0,True,Mekong Delta,1,2057000,3791.46,540,10591.12,338704.14
1,Bà Rịa–Vũng Tàu,2023-Q1,2023,1,35.333411,28.657948,112.670980,1558.0,7.0,True,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
2,Bình Dương,2023-Q1,2023,1,31.278105,25.175450,107.479716,747.0,11.0,True,Southeast,2,2564000,3663.54,901,10233.79,327276.61


## Summary

- Input: 24.7M raw NDT7 broadband test records for Vietnam (2023-2025)
- Output: province x quarter aggregates, tile-binned at Ookla's zoom-16 resolution, same
  `is_reliable` threshold as every Ookla country notebook
- Ready for: `notebooks/comparison/ookla_vs_ndt7_vietnam.ipynb` (Ookla-vs-NDT7 cross-validation for Vietnam)